# XGBoost 再学習ノートブック v6 — リーク修正 + base_margin パリティ対応

**このノートは上から順に全セルを実行するだけで完了します。**（所要 30〜60分）

## v5 からの変更点

| 内容 | 効果 |
|---|---|
| 学習データのリーク3系統を除去 | `f_early_speed`(実測ペース) / `running_style`(実測脚質) / `member_level`(未来参照) |
| `f_pred_gap_*` 5列を削除 | 「乖離は繰り返す」前提が実データで否定された（相関 -0.066） |
| `corner_all` を学習側にも供給 | `f_corner_position_change` が定数だったのを修正 |
| **base_margin に実測ドリフトを注入** | 学習=確定人気 / 推論=朝の薄い人気 というパリティ違反を是正 |

## 実行前の確認

- Colab の「シークレット」に `GITHUB_PAT` が登録されていること（最終セルのpushで使用）
- ランタイムは CPU で可

⚠ セル4の自動検証が **FAIL** を出したら、そこで止めて内容を確認してください。
その先に進むと壊れたモデルを本番へ反映してしまいます。


In [ ]:
# == セル1: セットアップ ===================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'BASE_DIR: {BASE_DIR}')


In [ ]:
# == セル2: src/ 強制アップデート（GitHub最新コードを取得）==================
# ⚠ v5から shap_explain.py / rank_matrix_filter.py を追加（v5では欠落しており
#    Colab側が古いままになっていた）
import urllib.request, time as _time

BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())

files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py', 'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py', 'src/tools/rescrape_history.py',
    'src/tools/build_training_data.py', 'src/tools/train_xgb.py',
    'src/tools/calibrate_xgb.py', 'src/tools/generate_style_advantage.py',
    'src/tools/train_pace_model.py', 'src/tools/shap_diagnosis.py',
    'src/features/engine.py', 'src/features/speed_index.py',
    'src/features/horse_type.py', 'src/features/error_tags.py',
    'src/features/shap_explain.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py', 'src/betting/ev_filter.py',
    'src/betting/app_json.py', 'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py', 'src/betting/rating_calibration.py',
    'src/betting/payout_estimator.py', 'src/betting/dual_model.py',
    'src/betting/bet_optimizer.py', 'src/betting/shadow.py',
    'src/betting/rank_matrix_filter.py',
]
data_files = [
    'data/course_profiles.json', 'data/course_distance_profiles.json',
    'data/note_schema.json',
]

_failed = []
for rel in files + data_files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for _retry in range(3):
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
            print(f'OK   {rel}')
            break
        except Exception as _e:
            if _retry < 2:
                _time.sleep(2 ** _retry)
            else:
                print(f'FAIL {rel}: {_e}')
                _failed.append(rel)

# モジュールキャッシュを破棄して最新を読み込ませる
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

assert not _failed, f'取得に失敗したファイルがあります: {_failed}'

# 今回の修正が入ったコードか確認（古いキャッシュのまま進むのを防ぐ）
import inspect
from src.tools.train_xgb import train_xgb, load_popularity_drift
from src.tools.build_training_data import build_training_data
assert 'simulate_serving_popularity' in inspect.signature(train_xgb).parameters, \
    'train_xgb が古い。強制アップデートが効いていません'
import src.tools.build_training_data as _btd
assert 'corner_all' in inspect.getsource(_btd), 'build_training_data が古い'
print('\n✅ 最新コードの取得を確認しました')


In [ ]:
# == セル3: GitHub→Drive データマージ（history.db の最新化）================
# 週次ワークフローがGitHub側の history.db を更新しているので取り込む。
# race_id をキーにした INSERT OR IGNORE なので既存データは壊れない。
import sqlite3, urllib.request, os

os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
gh_db = f'{BASE_DIR}/data/_history_github.db'
url = 'https://media.githubusercontent.com/media/hanagenuku/keiba_ai/main/data/history.db'
urllib.request.urlretrieve(url, gh_db)
print(f'GitHub history.db 取得: {os.path.getsize(gh_db):,} bytes')

local_db = f'{BASE_DIR}/data/history.db'

# Drive側のスキーマをGitHub側に揃える（空リストでALTER TABLEだけ適用）
from src.utils.db import save_history_db
save_history_db([], base_dir=BASE_DIR)

def merge(table):
    c = sqlite3.connect(local_db)
    c.execute(f"ATTACH DATABASE '{gh_db}' AS gh")
    loc = {r[1] for r in c.execute(f'PRAGMA table_info({table})')}
    rem = {r[1] for r in c.execute(f'PRAGMA table_info(gh.{table})')}
    common = sorted(loc & rem)
    only_gh = sorted(rem - loc)
    if only_gh:
        print(f'  ⚠ {table}: GitHub側にしか無い列（今回は取り込まず）: {only_gh}')
    before = c.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    cols = ','.join(f'"{x}"' for x in common)
    c.execute(f'INSERT OR IGNORE INTO {table} ({cols}) SELECT {cols} FROM gh.{table}')
    c.commit()
    after = c.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table}: {before:,} → {after:,} (+{after-before:,})')
    c.close()

for t in ('race_history', 'horse_history'):
    merge(t)

# keiba.db も取得（base_margin ドリフト分布の作成に odds_snapshots が要る）
try:
    urllib.request.urlretrieve(
        'https://media.githubusercontent.com/media/hanagenuku/keiba_ai/main/data/keiba.db',
        f'{BASE_DIR}/data/keiba.db')
    k = sqlite3.connect(f'{BASE_DIR}/data/keiba.db')
    n = k.execute('SELECT COUNT(*) FROM odds_snapshots').fetchone()[0]
    k.close()
    print(f'\nkeiba.db 取得: odds_snapshots {n:,} 行')
except Exception as e:
    print(f'\n⚠ keiba.db 取得失敗（ドリフト注入はスキップされます）: {e}')


In [ ]:
# == セル4: 学習データ再生成 + 修正が効いているかの自動検証 ================
import importlib, sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.features.speed_index import rebuild_speed_index_cache
try:
    rebuild_speed_index_cache(BASE_DIR)
    print('speed_index キャッシュ再構築 完了\n')
except Exception as e:
    print(f'speed_index 再構築スキップ: {e}\n')

from src.tools.build_training_data import build_training_data
build_training_data(BASE_DIR)

# ── 自動検証: 今回の修正がすべて反映されているか ──
import pandas as pd
df = pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
cols = list(df.columns)
checks = []

checks.append(('member_level 系が消えている',
               len([c for c in cols if 'member_level' in c]) == 0))
checks.append(('pred_gap 系が消えている',
               len([c for c in cols if 'pred_gap' in c]) == 0))
checks.append(('f_early_speed がリークしていない（定数36.0）',
               set(df['f_early_speed'].dropna().unique()) == {36.0}))
checks.append(('f_corner_position_change が定数でない（corner_all供給OK）',
               df['f_corner_position_change'].nunique() > 1))

print(f'\n{"="*60}\n学習データ検証: {len(df):,}行 × {len(cols)}列\n{"="*60}')
ok = True
for name, passed in checks:
    print(f'  {"PASS" if passed else "FAIL"}  {name}')
    ok &= passed
assert ok, '検証に失敗しました。ここで止めて内容を確認してください。'
print('\n✅ すべての修正が反映されています')


In [ ]:
# == セル5: 残差モデル再学習（base_margin ドリフト注入ON）==================
# simulate_serving_popularity=True（既定）で、確定人気に実測ドリフトを注入し
# 「推論時に届く朝の人気」と同じ情報量で学習する。
import sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.tools.train_xgb import train_xgb, load_popularity_drift

drift = load_popularity_drift(BASE_DIR)
if drift is None:
    print('⚠ ドリフト標本が不足しています（odds_snapshots が薄い）。')
    print('  従来どおり確定人気で学習します。数週間データが溜まってから再実行を推奨。')
else:
    print(f'ドリフト分布: {len(drift)-1}人気分 / 標本 {len(drift["_all"]):,} 件\n')

res_new = train_xgb(BASE_DIR, residual=True, simulate_serving_popularity=True)
print(f'\n残差モデル(ドリフト注入ON) Val AUC = {res_new["auc"]:.4f}')


In [ ]:
# == セル6: 比較（ドリフト注入 ON vs OFF）==================================
# 効果を数値で確認する。OFF側は比較専用で本番には反映しない。
import shutil, os

# ON側の成果物を退避（OFF学習で上書きされるため）
for f in ['xgb_fukusho_model_residual.pkl', 'xgb_feature_cols_residual.json']:
    p = f'{BASE_DIR}/data/{f}'
    if os.path.exists(p):
        shutil.copy2(p, p + '.drift_on')

res_off = train_xgb(BASE_DIR, residual=True, simulate_serving_popularity=False)

# ON側を復元
for f in ['xgb_fukusho_model_residual.pkl', 'xgb_feature_cols_residual.json']:
    p = f'{BASE_DIR}/data/{f}'
    if os.path.exists(p + '.drift_on'):
        shutil.move(p + '.drift_on', p)

print(f'\n{"="*60}')
print(f'  ドリフト注入 OFF (従来): Val AUC = {res_off["auc"]:.4f}')
print(f'  ドリフト注入 ON  (今回): Val AUC = {res_new["auc"]:.4f}')
print(f'{"="*60}')
print('\n※ この Val AUC は「確定人気で評価」した値のため、ONの方が低く出るのが正常です。')
print('  ONの真価は、本番と同じ「朝の人気」で推論したときに現れます')
print('  （オフライン検証では 複勝AUC +0.004 / 単勝AUC +0.006 / 較正後logloss -0.015）。')


In [ ]:
# == セル7: 本番切替 =======================================================
# 残差モデルを本番ファイル名にコピーし、キャリブレータを再作成する。
import shutil, json, os, sys

# 旧モデルをバックアップ
for f in ['xgb_fukusho_model.pkl', 'xgb_feature_cols.json', 'xgb_calibrator.pkl']:
    src = f'{BASE_DIR}/data/{f}'
    if os.path.exists(src):
        shutil.copy2(src, f'{src}.bak_v6')
        print(f'  バックアップ: {f} → {f}.bak_v6')

shutil.copy2(f'{BASE_DIR}/data/xgb_fukusho_model_residual.pkl',
             f'{BASE_DIR}/data/xgb_fukusho_model.pkl')
shutil.copy2(f'{BASE_DIR}/data/xgb_feature_cols_residual.json',
             f'{BASE_DIR}/data/xgb_feature_cols.json')

with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    meta = json.load(f)
assert meta.get('residual') is True, 'residual フラグがありません'
n_feat = len(meta['feature_cols'])
assert not [c for c in meta['feature_cols'] if 'member_level' in c or 'pred_gap' in c], \
    '削除したはずの列がモデルに残っています'
print(f'\n  residual={meta["residual"]}  特徴量数={n_feat}  Val AUC={meta.get("val_auc", 0):.4f}')

# キャリブレータ再作成（特徴量が変わったので必須）
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]
try:
    from src.tools.calibrate_xgb import run_xgb_calibration
    run_xgb_calibration(BASE_DIR)
    print('\n  キャリブレーション再作成 完了')
except Exception as e:
    print(f'\n  ⚠ キャリブレーション失敗: {e}')
    print('     （残差モデル非対応の既知問題。予測順位には影響しませんが、')
    print('       複勝確率の表示がやや古い較正のままになります）')


In [ ]:
# == セル8: 統合テスト =====================================================
# 本番と同じ経路で1レース分の推論を通し、壊れていないことを確認する。
import sys, sqlite3
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.features.engine import init_engine, calc_all
import src.features.engine as eng

init_engine(BASE_DIR)
print(f'  _XGB_RESIDUAL = {eng._XGB_RESIDUAL}  (True であること)')
assert eng._XGB_RESIDUAL is True, '残差モデルとして認識されていません'

# history.db から実在の1レースを取り出して推論
c = sqlite3.connect(f'{BASE_DIR}/data/history.db')
c.row_factory = sqlite3.Row
rid = c.execute('SELECT race_id FROM race_history ORDER BY date DESC LIMIT 1').fetchone()[0]
rows = c.execute('SELECT * FROM horse_history WHERE race_id=? ORDER BY horse_num', (rid,)).fetchall()
rr = c.execute('SELECT * FROM race_history WHERE race_id=?', (rid,)).fetchone()
c.close()

from src.scraper.jra_scraper import get_history_from_db
hist_path = f'{BASE_DIR}/data/history.db'
horses = []
for r in rows:
    horses.append({
        'name': r['horse_name'], 'horse_num': r['horse_num'],
        'post_position': r['horse_num'], 'jockey': r['jockey'] or '',
        'trainer': r['trainer'] or '', 'sex': r['sex'] or '牡',
        'age': r['age'] or 4, 'weight_load': r['weight_load'] or 56.0,
        'win_odds': None, 'popularity': r['popularity'] or 0,
        'history': get_history_from_db(r['horse_name'], hist_path),
    })

# calc_all は race 辞書ひとつを受け取り、出走馬は race['horses'] に入れる
race = {'race_id': rid, 'id': rid, 'date': rr['date'],
        'racecourse': rr['racecourse'], 'distance': rr['distance'],
        'surface': rr['surface'], 'track_condition': rr['track_condition'] or '良',
        'race_class': rr['race_class'] or '1勝クラス', 'race_num': rr['race_num'],
        'horses': horses}

scored = calc_all(race)
tot = sum(h.get('win_prob', 0) for h in scored)
print(f'\n  レース {rid}: {len(scored)}頭')
print(f'  win_prob 合計 = {tot:.4f}  (1.0 付近であること)')
assert 0.95 < tot < 1.05, 'win_prob が正規化されていません'
assert any(h.get('ability_margin') is not None for h in scored), \
    'ability_margin が None＝残差モデル経路を通っていません'
for h in sorted(scored, key=lambda x: -x.get('win_prob', 0))[:3]:
    print(f'    {h.get("name","?"):16s} win={h.get("win_prob",0)*100:5.1f}% '
          f'cal={h.get("cal_prob",0):.3f} ability={h.get("ability_margin")}')
print('\n✅ 統合テスト通過')


In [ ]:
# == セル9: GitHub main にプッシュ =========================================
import requests, base64, json as _json, os as _os
from google.colab import userdata

GITHUB_PAT = userdata.get('GITHUB_PAT')
REPO = 'hanagenuku/keiba_ai'
FILES = [
    'data/xgb_fukusho_model.pkl', 'data/xgb_feature_cols.json',
    'data/xgb_calibrator.pkl', 'data/pace_model.pkl',
    'data/jockey_pace_stats.json', 'data/speed_index_cache.pkl',
    'data/xgb_fukusho_model_residual.pkl', 'data/xgb_feature_cols_residual.json',
]

def push_file(relpath, pat, message):
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=h, params={'ref': 'main'})
    sha = r.json().get('sha') if r.status_code == 200 else None
    with open(f'{BASE_DIR}/{relpath}', 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    payload = {'message': message, 'content': content, 'branch': 'main'}
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=h, json=payload)
    ok = r.status_code in (200, 201)
    print(f'{"OK" if ok else f"NG({r.status_code})"} {relpath}')
    if not ok:
        print(f'  -> {r.json().get("message","")}')
    return ok

with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    meta = _json.load(f)
n = len(meta['feature_cols'])
msg = f'model: retrain v6 {n}feat AUC={meta.get("val_auc",0):.4f} (residual+drift)'
print(f'コミットメッセージ: {msg}\n')

pushed = sum(1 for rel in FILES
             if _os.path.exists(f'{BASE_DIR}/{rel}') and push_file(rel, GITHUB_PAT, msg))
print(f'\n完了: {pushed}/{len(FILES)} ファイルをpush')
print('\n>>> 次回の週末ワークフローから新モデルで予想が生成されます。')
print('    数週間後に data/kpi_weekly.json の delta を確認してください。')
